# One Health Disease Intelligence: Ingestion and Parameter Validation

This notebook implements the first step of our early warning disease intelligence workflow. We focus on **Harness Engineering** and **Context Engineering**:
1. **Ingesting meteorological and veterinary datasets** using the `DataIngestionAgent`.
2. **Applying strict biophysical range validation** to 18 environmental parameters (NDVI, LST, Soil Moisture, etc.) to sanitize sensor noise.
3. **Aggregating baseline reporting data** across our 19 priority pathogens using the `DiseaseInformaticsEngine`.

In [ ]:
# 1. Initialize Python Path for local module imports
import os
import sys
import pandas as pd
import numpy as np

# Ensure the 'src/' directory is in our path so we can import local modules
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../src")))
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

print("System path initialized successfully.")

## Step 2: Initialize Raw Source Files
We verify that the raw directory contains our target livestock disease case records and satellite meteorological observations.

In [ ]:
raw_vet_path = "../data/raw/veterinary_records.csv"
raw_climate_path = "../data/raw/climate_layers.csv"

# Check files
if os.path.exists(raw_vet_path) and os.path.exists(raw_climate_path):
    print("[SUCCESS] Source datasets located inside raw/ data directory.")
else:
    print("[ERROR] Missing raw data files. Execute run_pipeline.py from root first to auto-generate sample data.")

## Step 3: Run the Ingestion Validation Pipeline
We load our datasets. The `DataIngestionAgent` enforces Role-Based Access Control (RBAC) and clips any anomalous sensor reading (such as an NDVI index outside the physical bounds of `[-1.0, 1.0]`) directly back to realistic biophysical thresholds.

In [ ]:
from agents.data_ingestion_agent import DataIngestionAgent

# Instantiate Ingestion Agent under authenticated role
ingestion_agent = DataIngestionAgent(user_role="vet_lab_staff")

# Load datasets
vet_df = ingestion_agent.ingest_veterinary_records(raw_vet_path)
climate_df = ingestion_agent.ingest_climate_data(raw_climate_path)

print(f"Loaded {len(vet_df)} active veterinary records.")
print(f"Loaded {len(climate_df)} validated environmental data matrices.")

## Step 4: Perform Data Profiling (9 Dropdown Informatics Views)
Using the `DiseaseInformaticsEngine`, we aggregate our baseline datasets into the structured, regional profiling tables required by our dashboard's informatics views.

In [ ]:
from agents.disease_informatics_engine import DiseaseInformaticsEngine

# Perform an inner join on spatial identifier (district & region)
merged_df = pd.merge(vet_df, climate_df, on=["district", "region"], how="inner")

# Save processed master table for downstream modeling
master_output_path = "../data/processed/cleaned_master_dataset.csv"
os.makedirs(os.path.dirname(master_output_path), exist_ok=True)
merged_df.to_csv(master_output_path, index=False)
print(f"Saved cleaned master table to: {master_output_path}")

# Initialize Informatics Engine
informatics = DiseaseInformaticsEngine(merged_df)

# Extract specific reporting views
zoonotic_incidences = informatics.get_zoonotic_disease_incidences()
region_metrics = informatics.get_region_wise_metrics()
outbreak_details = informatics.compile_informatics_report()

print(f"\nTotal active national zoonotic cases: {zoonotic_incidences['total_cases'].sum()}")
print(f"Highest disease burden region: {outbreak_details['highest_burden_region']}")

## Step 5: Render Color-Coded Monthly Reporting Completion Grid
Finally, we run our `OutbreakStatusTracker` utility to render our styled, color-coded data-completeness grid showing which regional states have submitted their clinical logs.

In [ ]:
from utils import OutbreakStatusTracker

tracker = OutbreakStatusTracker(target_year=2026)
raw_compliance_df = tracker.generate_compliance_matrix()

# Filter for specific target regions of interest
searched_df = raw_compliance_df[raw_compliance_df["State Name"].str.contains("South|Afar|Somali", case=False)]

# Render styled, color-coded HTML table directly inside cell
styled_grid = tracker.apply_dashboard_styles(searched_df)
styled_grid